In [1]:
import os
os.chdir('..')
print(os.getcwd())

d:\pythonProject\IC Lab\Gait_analysis\pyskl


In [ ]:
import re, gc
import numpy as np
import subprocess
from colab.tools import prinfo, config
from colab.SSO import SSO
import torch


In [ ]:
sso_setting = {
        789 : {'cg' : 0.7, 'cp' : 0.8, 'cw': 0.9},
        189 : {'cg' : 0.1, 'cp' : 0.8, 'cw': 0.9}, 
        129 : {'cg' : 0.1, 'cp' : 0.2, 'cw': 0.9}, 
        123 : {'cg' : 0.1, 'cp' : 0.2, 'cw': 0.3}, 
    }

for setting in sso_setting.keys():
    for split in [8]:
        params_range = {
            'num_init'      : (1, 4),
            'base_channel'  : (64, 256),
            'stride_init'   : (1, 3),
            'tkernel_init'  : (3, 5, 7),
            
            'act'           : (0, 3),           # activate function  
            'opt'           : (0, 2),           # optimizer

            'dropout_bk'    : (15, 30),         # dropout in block *10-2
            'dropout_fc'    : (15, 30),         # dropout in fc *10-2

            'lr'            : (1, 100),         # learning rate *10-3
            'weight_decay'  : (0, 100),         # weight_decay *10-4
            'momentum'      : (50, 99),          # momentum * 10-2
            'margin'        : (20, 100),          # margin in Triplet Loss *10-2
            'lambda_val'    : (0, 90),          # The ratio of Triplet Loss compare to Cross Entropy *10-2
        }

        # 設定初始解
        params = {
            'data'          : 'drunk',          # 我自己讀特定資料集的方式
            'batch'         : 128,              # 可以調整~~
            'epoch'         : 80,               # 可以調整~~
            'feature'       : 'j',              # 我自己讀特定資料集的方式
            'split'         : split,

            'base_channel'  : config[split]['base_channel'],           # output_channel in init_layer
            'num_init'      : config[split]['num_init'],
            'stride_init'   : 1,
            'tk_init'       : 3,

            'num_in'        : 0,                # num_layers_input_stream 固定捨棄後面兩個stream
            'num_main'      : 0,                # num_layers_main_stream 固定捨棄後面兩個stream

            'act'           : 0,                # activate function
            'opt'           : 0,                # optimizer
            
            'dropout_bk'    : 0,                # dropout in block
            'dropout_fc'    : 0,                # dropout in fc

            'lr'            : 0.1,              # learning rate
            'weight_decay'  : 0.0005,           # weight_decay
            'momentum'      : 0.9,              # momentum
            'margin'        : config[split]['margin'],              # margin in Triplet Loss
            'lambda_val'    : config[split]['lambda'],              # The ratio of Triplet Loss compare to Cross Entropy
        }
                                                                                                                
        # 不用動
        def get_param(X, keys=params_range.keys(), params=params):
            """
            將搜尋空間中編碼過的數值 X 映射回實際的模型參數設定，統一保留4位小數。

            Parameters:
                X (np.ndarray): 搜尋演算法產出的參數值列表。
                keys (list): 與 X 對應的參數名稱。
                params (dict): 預設參數字典。

            Returns:
                dict: 還原為實際值後的參數字典。
            """
            param = params.copy()

            for i, key in enumerate(keys):
                val = X[i]
                boundary = params_range[key]

                # 統一保留 4 位小數的縮放邏輯
                if key in ['dropout_bk', 'dropout_fc']:
                    param[key] = round(val * 1e-2, 4)
                elif key == 'lr':
                    param[key] = round(val * 1e-3, 4)
                elif key == 'weight_decay':
                    param[key] = round(val * 1e-4, 4)
                elif key == 'momentum':
                    param[key] = round(val * 1e-2, 4)
                elif key == 'margin':
                    param[key] = round(val * 1e-2, 4)
                elif key == 'lambda_val':
                    param[key] = round(val * 1e-2, 4)
                elif (
                    isinstance(boundary, (tuple, list)) and
                    all(isinstance(b, int) for b in boundary)
                ):
                    param[key] = int(val)
                else:
                    param[key] = val

            return param

        def fitness(X):
            # **釋放 GPU 記憶體**
            torch.cuda.empty_cache()
            
            param = get_param(X=X)
            param_args = " ".join([f"--{key} {str(value).strip()}" for key, value in param.items()])
            command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu --mt {param_args}"
            
            try:
                # 使用 `!{command}` 在 Jupyter Notebook / Colab 內執行
                output = !{command}
                # print(output)
                # 解析輸出
                train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, \
                test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, log = prinfo(output)

            except Exception as e:
                print(f"❌ 發生錯誤: {e}")
                # 確保所有變數有值，避免 `NoneType` 出錯
                train_cost = train_time = test_time = 0
                val_acc_4c = val_acc_3c = val_acc_2c = 0
                test_acc_4c = test_acc_3c = test_acc_2c = 0
                f1 = pre = rec = 0
                log = ""

            # 確保 `NoneType` 不影響計算
            fitness_value = (val_acc_4c or 0)

            # 確保 `log` 不為 `None`
            log = log if log is not None else ""

            record_message = {
                'train_cost': train_cost,
                'train_time': train_time,
                'test_time': test_time,
                'val_acc_4c': val_acc_4c,
                'val_acc_3c': val_acc_3c,
                'val_acc_2c': val_acc_2c,
                'test_acc_4c': test_acc_4c,
                'test_acc_3c': test_acc_3c,
                'test_acc_2c': test_acc_2c,
                'f1': f1,
                'precision': pre,
                'recall': rec,
                'log': log
            }
            
            return fitness_value, record_message

        for i in range(5):
            save_name = f"{params['data']}_{params['split']}_{setting}_run{i}"
            SSO_searcher = SSO(
                Ngen=15,                                # 世代數
                Nsol=10,                                # 解的數量
                Cg=sso_setting[setting]['cg'],          # 全局最佳解的權重
                Cp=sso_setting[setting]['cp'],          # 個體最佳解的權重
                Cw=sso_setting[setting]['cw'],          # 隨機解的權重
                save_name=save_name,                    # 記錄檔儲存前綴
                fitness=fitness,                        # 適應度函數
                base_param=params,                      # 基礎參數 (可選)
                boundary=params_range,                  # 支持 tuple 或 list (詳細說明請參照前述)
                direction="maximize",                   # 優化方向 ('maximize', 'minimize')
                )

            # 直接運行
            SSO_searcher.run()